In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr

from sklearn.linear_model    import Ridge, ElasticNet
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (mean_squared_error, mean_absolute_error,
                                     r2_score, mean_absolute_percentage_error)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.inspection      import permutation_importance
from sklearn.base            import clone

import os

OUT = "/content/"
os.makedirs(OUT, exist_ok=True)

COLORS = {
    "ridge"     : "#2166AC",
    "elastic"   : "#D6604D",
    "actual"    : "#1A1A2E",
    "train_bg"  : "#EBF5FB",
    "pred_bg"   : "#FEF9E7",
    "grid"      : "#DDDDDD",
}
plt.rcParams.update({
    "font.family"       : "DejaVu Serif",
    "font.size"         : 11,
    "axes.titlesize"    : 13,
    "axes.labelsize"    : 12,
    "axes.linewidth"    : 1.2,
    "axes.grid"         : True,
    "grid.color"        : COLORS["grid"],
    "grid.linewidth"    : 0.6,
    "grid.linestyle"    : "--",
    "legend.framealpha" : 0.9,
    "legend.fontsize"   : 10,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "figure.dpi"        : 150,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
    "savefig.facecolor" : "white",
})

MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

print("="*70)
print("  DENGUE CLIMATE REGRESSION ANALYSIS — BANGLADESH (2008–2025)")
print("  LEAKAGE-FREE PIPELINE  |  Train: 2008–2024  |  Test: 2025 Jan–Oct")
print("="*70)

  DENGUE CLIMATE REGRESSION ANALYSIS — BANGLADESH (2008–2025)
  LEAKAGE-FREE PIPELINE  |  Train: 2008–2024  |  Test: 2025 Jan–Oct


## 1.  DATA LOADING & FEATURE ENGINEERING

In [ ]:
df = pd.read_csv("/content/Dengue_Climate_Bangladesh.csv")

df["DATE"]     = pd.to_datetime(df[["YEAR","MONTH"]].assign(DAY=1))
df             = df.sort_values("DATE").reset_index(drop=True)
df["TIME_IDX"] = np.arange(len(df))          # monotonic trend variable, known in advance

# ── Engineered features (all computed from same-row or PAST rows only)
df["TEMP_RANGE"]     = df["MAX"] - df["MIN"]
df["TEMP_MEAN"]      = (df["MAX"] + df["MIN"]) / 2
df["HEAT_INDEX"]     = (df["TEMP_MEAN"] * df["HUMIDITY"]) / 100
df["RAIN_LOG"]       = np.log1p(df["RAINFALL"])
df["SIN_MONTH"]      = np.sin(2 * np.pi * df["MONTH"] / 12)
df["COS_MONTH"]      = np.cos(2 * np.pi * df["MONTH"] / 12)
df["DENGUE_LOG"]     = np.log1p(df["DENGUE"])

# ── Lag features (strictly backward-looking — .shift() never sees future rows)
for lag in [1, 2, 3]:
    df[f"DENGUE_LAG{lag}"]  = df["DENGUE_LOG"].shift(lag)
    df[f"RAIN_LAG{lag}"]    = df["RAIN_LOG"].shift(lag)
    df[f"HUMID_LAG{lag}"]   = df["HUMIDITY"].shift(lag)

df = df.dropna().reset_index(drop=True)

FEATURES = [
    "TEMP_MEAN","TEMP_RANGE","HEAT_INDEX",
    "HUMIDITY","RAIN_LOG","RAINFALL",
    "SIN_MONTH","COS_MONTH","TIME_IDX",
    "DENGUE_LAG1","DENGUE_LAG2","DENGUE_LAG3",
    "RAIN_LAG1","RAIN_LAG2","HUMID_LAG1",
]
TARGET = "DENGUE_LOG"

print(f"\n✔  Dataset loaded  : {df.shape[0]} observations × {df.shape[1]} columns")
print(f"✔  Feature set     : {len(FEATURES)} predictors")


✔  Dataset loaded  : 211 observations × 25 columns
✔  Feature set     : 15 predictors


## 2.  TRAIN / TEST SPLIT  (train: 2008-2024 | predict: 2025 Jan–Oct)

In [ ]:
train_mask = df["YEAR"] <= 2024
test_mask  = df["YEAR"] == 2025

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

# pipeline/CV fold
X_train = df_train[FEATURES].values
y_train = df_train[TARGET].values
X_test  = df_test[FEATURES].values
y_test  = df_test[TARGET].values

print(f"\n✔  Training set    : {len(df_train)} months  ({df_train['YEAR'].min()}–{df_train['YEAR'].max()})")
print(f"✔  Prediction set  : {len(df_test)} months  (2025 Jan–Oct)")


✔  Training set    : 201 months  (2008–2024)
✔  Prediction set  : 10 months  (2025 Jan–Oct)


## 3.  HYPERPARAMETER TUNING via LEAKAGE-FREE TIME-SERIES CROSS-VALIDATION

In [ ]:
tscv = TimeSeriesSplit(n_splits=10)

alphas    = np.logspace(-4, 4, 100)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge",  Ridge()),
])
grid_ridge = GridSearchCV(
    ridge_pipe,
    param_grid={"ridge__alpha": alphas},
    cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
grid_ridge.fit(X_train, y_train)
best_alpha_ridge = grid_ridge.best_params_["ridge__alpha"]
ridge = grid_ridge.best_estimator_          # Pipeline refit on ALL of X_train (test never touched)

enet_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("enet",   ElasticNet(max_iter=20000)),
])
grid_enet = GridSearchCV(
    enet_pipe,
    param_grid={"enet__alpha": np.logspace(-4, 4, 40), "enet__l1_ratio": l1_ratios},
    cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
grid_enet.fit(X_train, y_train)
best_alpha_enet = grid_enet.best_params_["enet__alpha"]
best_l1         = grid_enet.best_params_["enet__l1_ratio"]
enet = grid_enet.best_estimator_

print(f"\n✔  Ridge      best α = {best_alpha_ridge:.4f}")
print(f"✔  ElasticNet best α = {best_alpha_enet:.4f},  l1_ratio = {best_l1:.2f}")


✔  Ridge      best α = 5.8570
✔  ElasticNet best α = 0.0744,  l1_ratio = 1.00


## 4.  PREDICTIONS  (log-space → original scale)

In [ ]:
def predict_orig(model, X):
    return np.expm1(model.predict(X))

ridge_train_pred  = predict_orig(ridge, X_train)
ridge_test_pred   = predict_orig(ridge, X_test)
enet_train_pred   = predict_orig(enet,  X_train)
enet_test_pred    = predict_orig(enet,  X_test)

y_train_orig      = np.expm1(y_train)
y_test_orig       = np.expm1(y_test)

## 5.  METRICS

In [ ]:
def compute_metrics(y_true, y_pred, label):
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true + 1, y_pred + 1) * 100
    corr, pval = pearsonr(y_true, y_pred)
    return {"Model": label, "RMSE": rmse, "MAE": mae,
            "R²": r2, "MAPE(%)": mape, "Pearson r": corr, "p-value": pval}

metrics_rows = [
    compute_metrics(y_train_orig, ridge_train_pred, "Ridge — Train"),
    compute_metrics(y_test_orig,  ridge_test_pred,  "Ridge — Test (2025)"),
    compute_metrics(y_train_orig, enet_train_pred,  "ElasticNet — Train"),
    compute_metrics(y_test_orig,  enet_test_pred,   "ElasticNet — Test (2025)"),
]
metrics_df = pd.DataFrame(metrics_rows)

# ── 10-Fold (TimeSeriesSplit) CV RMSE on training set, scaler refit per fold
cv_ridge = -cross_val_score(clone(ridge), X_train, y_train,
                             cv=tscv, scoring="neg_root_mean_squared_error")
cv_enet  = -cross_val_score(clone(enet), X_train, y_train,
                             cv=tscv, scoring="neg_root_mean_squared_error")

print("\n" + "─"*70)
print("  PERFORMANCE METRICS")
print("─"*70)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  10-Fold (TimeSeriesSplit) CV RMSE (log-scale):")
print(f"    Ridge      : {cv_ridge.mean():.4f} ± {cv_ridge.std():.4f}")
print(f"    ElasticNet : {cv_enet.mean():.4f}  ± {cv_enet.std():.4f}")

metrics_df.to_csv(f"{OUT}metrics_summary.csv", index=False)


──────────────────────────────────────────────────────────────────────
  PERFORMANCE METRICS
──────────────────────────────────────────────────────────────────────
                   Model      RMSE       MAE      R²  MAPE(%)  Pearson r  p-value
           Ridge — Train 5003.0580 1322.9909  0.7959 114.5611     0.9384   0.0000
     Ridge — Test (2025) 9000.6825 5222.7688 -1.1473  65.9896     0.8821   0.0007
      ElasticNet — Train 5547.4948 1475.6127  0.7491 123.4636     0.9327   0.0000
ElasticNet — Test (2025) 6442.2205 3453.9784 -0.1001  52.2296     0.8730   0.0010

  10-Fold (TimeSeriesSplit) CV RMSE (log-scale):
    Ridge      : 1.0501 ± 0.2168
    ElasticNet : 1.0044  ± 0.2055


## 6.  FIGURE 1 — Exploratory Data Analysis (4-panel)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Figure 1. Exploratory Data Analysis — Dengue & Climate Variables\n"
             "Bangladesh, 2008–2025", fontsize=14, fontweight="bold", y=1.01)

ax = axes[0, 0]
ax.fill_between(df["DATE"], df["DENGUE"], alpha=0.2, color=COLORS["actual"])
ax.plot(df["DATE"], df["DENGUE"], color=COLORS["actual"], lw=1.3)
ax.set_title("(a) Monthly Dengue Cases")
ax.set_xlabel("Year"); ax.set_ylabel("Cases")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))

ax = axes[0, 1]
monthly_data = [df[df["MONTH"]==m]["DENGUE"].values for m in range(1,13)]
bp = ax.boxplot(monthly_data, patch_artist=True,
                medianprops=dict(color="black", lw=2))
cmap = plt.cm.RdYlBu_r
for i, patch in enumerate(bp["boxes"]):
    patch.set_facecolor(cmap(i / 11))
ax.set_xticklabels(MONTH_NAMES)
ax.set_title("(b) Seasonal Distribution of Dengue Cases")
ax.set_xlabel("Month"); ax.set_ylabel("Cases")

ax = axes[1, 0]
corr_cols = ["DENGUE","TEMP_MEAN","TEMP_RANGE","HUMIDITY","RAINFALL","RAIN_LOG"]
corr_labels = ["Dengue","Temp Mean","Temp Range","Humidity","Rainfall","Rainfall(log)"]
corr_mat = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, square=True, linewidths=0.5,
            xticklabels=corr_labels, yticklabels=corr_labels,
            cbar_kws={"shrink": 0.8})
ax.set_title("(c) Pearson Correlation Matrix")

ax = axes[1, 1]
ax2 = ax.twinx()
ax.bar(df["DATE"], df["RAINFALL"], width=20, alpha=0.4,
       color="#1ABC9C", label="Rainfall (mm)")
ax2.plot(df["DATE"], df["HUMIDITY"], color="#E74C3C", lw=1.5, label="Humidity (%)")
ax2.plot(df["DATE"], df["TEMP_MEAN"], color="#F39C12", lw=1.5, label="Temp Mean (°C)")
ax.set_xlabel("Year"); ax.set_ylabel("Rainfall (mm)")
ax2.set_ylabel("Humidity (%) / Temperature (°C)")
ax.set_title("(d) Climate Variables Over Time")
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, loc="upper left", fontsize=9)

plt.tight_layout()
fig.savefig(f"{OUT}Fig1_EDA.png")
plt.close()
print("\nFigure 1 saved  → Fig1_EDA.png")


✔  Figure 1 saved  → Fig1_EDA.png


## 7.  FIGURE 2 — Full Time-Series: Actual vs Predicted (Train + 2025 Forecast)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)
fig.suptitle("Figure 2. Dengue Case Prediction — Actual vs. Model Output\n"
             "Bangladesh 2008–2025 (Train) + 2025 Jan–Oct (Forecast)",
             fontsize=14, fontweight="bold")

all_dates  = df["DATE"].values
all_actual = np.expm1(df[TARGET].values)
train_dates = df_train["DATE"].values
test_dates  = df_test["DATE"].values

for i, (model_name, tr_pred, te_pred, color) in enumerate([
    ("Ridge Regression",    ridge_train_pred, ridge_test_pred, COLORS["ridge"]),
    ("ElasticNet Regression", enet_train_pred, enet_test_pred, COLORS["elastic"]),
]):
    ax = axes[i]
    ax.axvspan(pd.Timestamp("2008-01-01"), pd.Timestamp("2024-12-31"),
               alpha=0.06, color="steelblue", label="_nolegend_")
    ax.axvspan(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-11-01"),
               alpha=0.08, color="gold", label="_nolegend_")

    ax.plot(all_dates, all_actual, color=COLORS["actual"],
            lw=1.6, label="Actual Cases", zorder=3)
    ax.plot(train_dates, tr_pred, color=color,
            lw=1.4, linestyle="--", alpha=0.75, label=f"{model_name} (Train)", zorder=2)
    ax.plot(test_dates, te_pred, color=color,
            lw=2.5, linestyle="-", marker="o", markersize=6,
            label=f"{model_name} (2025 Forecast)", zorder=4)
    ax.plot(test_dates, y_test_orig, color="black",
            lw=2.0, linestyle="-", marker="s", markersize=6, alpha=0.9,
            label="Actual 2025", zorder=5)

    ax.set_ylabel("Dengue Cases", fontsize=11)
    ax.set_title(f"({chr(97+i)}) {model_name}", fontsize=12)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.legend(loc="upper left", fontsize=9.5, ncol=2)

    for d, pred, actual in zip(test_dates, te_pred, y_test_orig):
        mname = MONTH_NAMES[pd.Timestamp(d).month - 1]
        ax.annotate(f"{int(pred):,}", xy=(d, pred),
                    xytext=(0, 10), textcoords="offset points",
                    ha="center", fontsize=7.5, color=color, fontweight="bold")

    ax.text(pd.Timestamp("2015-06-01"), ax.get_ylim()[1]*0.88,
            "◀  Training Period (2008–2024)  ▶",
            ha="center", fontsize=9, color="steelblue", style="italic")
    ax.text(pd.Timestamp("2025-05-15"), ax.get_ylim()[1]*0.88,
            "2025\nForecast",
            ha="center", fontsize=9, color="goldenrod", style="italic")

axes[-1].set_xlabel("Date", fontsize=11)
plt.tight_layout()
fig.savefig(f"{OUT}Fig2_TimeSeries_Predictions.png")
plt.close()
print("Figure 2 saved  → Fig2_TimeSeries_Predictions.png")

✔  Figure 2 saved  → Fig2_TimeSeries_Predictions.png


## 8.  FIGURE 3 — 2025 Close-Up: Bar Chart Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Figure 3. 2025 Monthly Forecast vs Actual Dengue Cases (Jan–Oct)\n"
             "Bangladesh", fontsize=14, fontweight="bold")

months_2025 = [MONTH_NAMES[m-1] for m in df_test["MONTH"].values]
x = np.arange(len(months_2025))
w = 0.28

for i, (ax, model_name, te_pred, color) in enumerate(zip(
    axes,
    ["Ridge Regression", "ElasticNet Regression"],
    [ridge_test_pred, enet_test_pred],
    [COLORS["ridge"], COLORS["elastic"]],
)):
    bars_a = ax.bar(x - w/2, y_test_orig, width=w, label="Actual",
                    color=COLORS["actual"], alpha=0.85, edgecolor="white")
    bars_p = ax.bar(x + w/2, te_pred,     width=w, label="Predicted",
                    color=color, alpha=0.85, edgecolor="white")

    for j, (a, p) in enumerate(zip(y_test_orig, te_pred)):
        err_pct = abs(a - p) / (a + 1) * 100
        ax.text(j, max(a, p) + 500, f"{err_pct:.0f}%",
                ha="center", va="bottom", fontsize=8, color="dimgray")

    ax.set_xticks(x); ax.set_xticklabels(months_2025)
    ax.set_title(f"({chr(97+i)}) {model_name}", fontsize=12)
    ax.set_xlabel("Month (2025)"); ax.set_ylabel("Dengue Cases")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.legend()
    r2_val = r2_score(y_test_orig, te_pred)
    rmse_val = np.sqrt(mean_squared_error(y_test_orig, te_pred))
    ax.text(0.02, 0.97, f"R² = {r2_val:.3f}\nRMSE = {rmse_val:,.0f}",
            transform=ax.transAxes, va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.8))

plt.tight_layout()
fig.savefig(f"{OUT}Fig3_2025_Forecast_Bar.png")
plt.close()
print("Figure 3 saved  → Fig3_2025_Forecast_Bar.png")

✔  Figure 3 saved  → Fig3_2025_Forecast_Bar.png


## 9.  FIGURE 4 — Scatter: Actual vs Predicted (Train + Test)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle("Figure 4. Scatter Plots: Actual vs. Predicted Dengue Cases",
             fontsize=14, fontweight="bold")

combos = [
    ("Ridge — Training Set",      y_train_orig, ridge_train_pred, COLORS["ridge"],   "(a)"),
    ("Ridge — 2025 Forecast",     y_test_orig,  ridge_test_pred,  COLORS["ridge"],   "(b)"),
    ("ElasticNet — Training Set", y_train_orig, enet_train_pred,  COLORS["elastic"], "(c)"),
    ("ElasticNet — 2025 Forecast",y_test_orig,  enet_test_pred,   COLORS["elastic"], "(d)"),
]
for ax, (title, y_t, y_p, color, lbl) in zip(axes.flat, combos):
    ax.scatter(y_t, y_p, alpha=0.55, edgecolors=color, facecolors="none",
               s=50, lw=1.2, label="Data points")
    lo, hi = min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.2, label="Perfect fit")

    slope, intercept, r, p_val, _ = stats.linregress(y_t, y_p)
    x_fit = np.linspace(lo, hi, 200)
    ax.plot(x_fit, slope * x_fit + intercept, color=color,
            lw=1.8, label=f"OLS: y={slope:.2f}x+{intercept:.0f}")

    r2 = r2_score(y_t, y_p)
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    ax.set_title(f"{lbl} {title}", fontsize=11)
    ax.set_xlabel("Actual Cases"); ax.set_ylabel("Predicted Cases")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
    ax.text(0.04, 0.95,
            f"R² = {r2:.3f}\nRMSE = {rmse:,.0f}\nr = {r:.3f}",
            transform=ax.transAxes, va="top", fontsize=9.5,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85))
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(f"{OUT}Fig4_Scatter_ActualVsPred.png")
plt.close()
print("Figure 4 saved  → Fig4_Scatter_ActualVsPred.png")

✔  Figure 4 saved  → Fig4_Scatter_ActualVsPred.png


## 10.  FIGURE 5 — Residual Analysis (4-panel × 2 models)

In [ ]:
for model_name, y_p_tr, y_p_te, color, suffix in [
    ("Ridge Regression",     ridge_train_pred, ridge_test_pred, COLORS["ridge"],   "Ridge"),
    ("ElasticNet Regression",enet_train_pred,  enet_test_pred,  COLORS["elastic"], "ElasticNet"),
]:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(f"Figure 5{'a' if suffix=='Ridge' else 'b'}. "
                 f"Residual Diagnostics — {model_name}",
                 fontsize=13, fontweight="bold")

    res_train = y_train_orig - y_p_tr
    res_test  = y_test_orig  - y_p_te

    ax = axes[0, 0]
    ax.scatter(y_p_tr, res_train, alpha=0.45, color=color, s=30, edgecolors="none")
    ax.axhline(0, color="black", lw=1.2, ls="--")
    ax.set_xlabel("Fitted Values"); ax.set_ylabel("Residuals")
    ax.set_title("(a) Residuals vs Fitted (Train)")

    ax = axes[0, 1]
    stats.probplot(res_train, dist="norm", plot=ax)
    ax.get_lines()[0].set(markerfacecolor=color, markeredgecolor=color,
                           alpha=0.5, markersize=4)
    ax.get_lines()[1].set(color="black", lw=1.5)
    ax.set_title("(b) Normal Q-Q Plot (Train Residuals)")

    ax = axes[1, 0]
    ax.hist(res_train, bins=30, color=color, alpha=0.7, edgecolor="white",
            density=True)
    mu, sigma = res_train.mean(), res_train.std()
    x_norm = np.linspace(res_train.min(), res_train.max(), 200)
    ax.plot(x_norm, stats.norm.pdf(x_norm, mu, sigma),
            "k-", lw=2, label=f"N({mu:.0f},{sigma:.0f})")
    ax.set_xlabel("Residuals"); ax.set_ylabel("Density")
    ax.set_title("(c) Residual Distribution (Train)")
    ax.legend(fontsize=9)

    ax = axes[1, 1]
    bars = ax.bar(months_2025, res_test, color=[
        "#2ECC71" if r >= 0 else "#E74C3C" for r in res_test], edgecolor="white")
    ax.axhline(0, color="black", lw=1.2)
    ax.set_xlabel("Month (2025)"); ax.set_ylabel("Residual (Actual − Predicted)")
    ax.set_title("(d) Forecast Residuals — 2025 Jan–Oct")
    for bar, val in zip(bars, res_test):
        ax.text(bar.get_x() + bar.get_width()/2,
                val + (500 if val >= 0 else -800),
                f"{int(val):,}", ha="center", fontsize=8)

    plt.tight_layout()
    fig.savefig(f"{OUT}Fig5_{suffix}_Residuals.png")
    plt.close()
    print(f"Figure 5 saved  → Fig5_{suffix}_Residuals.png")

✔  Figure 5 saved  → Fig5_Ridge_Residuals.png
✔  Figure 5 saved  → Fig5_ElasticNet_Residuals.png


## 11.  FIGURE 6 — Feature Importance (Coefficients + Permutation)

In [ ]:
feat_labels = [
    "Temp Mean","Temp Range","Heat Index",
    "Humidity","Rain (log)","Rainfall",
    "sin(Month)","cos(Month)","Time Index",
    "Dengue Lag1","Dengue Lag2","Dengue Lag3",
    "Rain Lag1","Rain Lag2","Humid Lag1",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Figure 6. Feature Importance Analysis\nRidge & ElasticNet Models",
             fontsize=13, fontweight="bold")

for row, (model, model_name, color, step) in enumerate([
    (ridge, "Ridge Regression",     COLORS["ridge"],   "ridge"),
    (enet,  "ElasticNet Regression", COLORS["elastic"], "enet"),
]):
    coefs = model.named_steps[step].coef_
    order = np.argsort(np.abs(coefs))[::-1]

    ax = axes[row, 0]
    colors_bar = [color if c >= 0 else "#E74C3C" for c in coefs[order]]
    ax.barh([feat_labels[i] for i in order], coefs[order],
            color=colors_bar, edgecolor="white")
    ax.axvline(0, color="black", lw=1.2)
    ax.set_xlabel("Standardized Coefficient")
    ax.set_title(f"({'ac'[row]}) {model_name} — Coefficients")
    ax.invert_yaxis()

    ax = axes[row, 1]
    perm = permutation_importance(model, X_test, y_test,
                                   n_repeats=30, random_state=42)
    perm_means = perm.importances_mean
    perm_stds  = perm.importances_std
    perm_order = np.argsort(perm_means)[::-1]
    ax.barh([feat_labels[i] for i in perm_order],
            perm_means[perm_order],
            xerr=perm_stds[perm_order],
            color=color, alpha=0.8, edgecolor="white", capsize=3)
    ax.axvline(0, color="black", lw=1.2)
    ax.set_xlabel("Permutation Importance (MSE increase)")
    ax.set_title(f"({'bd'[row]}) {model_name} — Permutation Importance (Test)")
    ax.invert_yaxis()

plt.tight_layout()
fig.savefig(f"{OUT}Fig6_Feature_Importance.png")
plt.close()
print("✔  Figure 6 saved  → Fig6_Feature_Importance.png")

✔  Figure 6 saved  → Fig6_Feature_Importance.png


## 12.  FIGURE 7 — Regularization Path (alpha vs RMSE), leakage-free CV

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 7. Regularization Path — Cross-Validation RMSE vs Alpha",
             fontsize=13, fontweight="bold")

alpha_grid = np.logspace(-4, 4, 40)
ridge_cv_scores = []
for a in alpha_grid:
    pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=a))])
    scores = -cross_val_score(pipe, X_train, y_train,
                               cv=tscv, scoring="neg_root_mean_squared_error")
    ridge_cv_scores.append((scores.mean(), scores.std()))
ridge_means, ridge_stds = zip(*ridge_cv_scores)

ax = axes[0]
ax.semilogx(alpha_grid, ridge_means, color=COLORS["ridge"], lw=2)
ax.fill_between(alpha_grid,
                np.array(ridge_means) - np.array(ridge_stds),
                np.array(ridge_means) + np.array(ridge_stds),
                alpha=0.2, color=COLORS["ridge"])
ax.axvline(best_alpha_ridge, color="red", lw=1.8, ls="--",
           label=f"Best α = {best_alpha_ridge:.4f}")
ax.set_xlabel("Alpha (Regularization Strength) — log scale")
ax.set_ylabel("CV RMSE (log-scale)")
ax.set_title("(a) Ridge Regression")
ax.legend()

enet_cv_scores = []
for a in alpha_grid:
    pipe = Pipeline([("scaler", StandardScaler()),
                      ("enet", ElasticNet(alpha=a, l1_ratio=best_l1, max_iter=20000))])
    scores = -cross_val_score(pipe, X_train, y_train,
                               cv=tscv, scoring="neg_root_mean_squared_error")
    enet_cv_scores.append((scores.mean(), scores.std()))
enet_means, enet_stds = zip(*enet_cv_scores)

ax = axes[1]
ax.semilogx(alpha_grid, enet_means, color=COLORS["elastic"], lw=2)
ax.fill_between(alpha_grid,
                np.array(enet_means) - np.array(enet_stds),
                np.array(enet_means) + np.array(enet_stds),
                alpha=0.2, color=COLORS["elastic"])
ax.axvline(best_alpha_enet, color="red", lw=1.8, ls="--",
           label=f"Best α = {best_alpha_enet:.4f}  (l1={best_l1:.2f})")
ax.set_xlabel("Alpha (Regularization Strength) — log scale")
ax.set_ylabel("CV RMSE (log-scale)")
ax.set_title("(b) ElasticNet Regression")
ax.legend()

plt.tight_layout()
fig.savefig(f"{OUT}Fig7_Regularization_Path.png")
plt.close()
print("Figure 7 saved  → Fig7_Regularization_Path.png")

✔  Figure 7 saved  → Fig7_Regularization_Path.png


## 13.  FIGURE 8 — Year-wise Annual Aggregation: Predicted vs Actual

In [ ]:
df_plot = df_train.copy()
df_plot["Ridge_Pred"]  = ridge_train_pred
df_plot["ENet_Pred"]   = enet_train_pred

annual = df_plot.groupby("YEAR").agg(
    Actual  =("DENGUE","sum"),
    Ridge   =("Ridge_Pred","sum"),
    ENet    =("ENet_Pred","sum"),
).reset_index()

test_annual = pd.DataFrame({
    "YEAR"  : [2025],
    "Actual": [y_test_orig.sum()],
    "Ridge" : [ridge_test_pred.sum()],
    "ENet"  : [enet_test_pred.sum()],
})
annual_full = pd.concat([annual, test_annual], ignore_index=True)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(annual_full))
w = 0.28
ax.bar(x - w,   annual_full["Actual"], w, label="Actual",
       color=COLORS["actual"], alpha=0.85, edgecolor="white")
ax.bar(x,       annual_full["Ridge"],  w, label="Ridge Predicted",
       color=COLORS["ridge"],  alpha=0.85, edgecolor="white")
ax.bar(x + w,   annual_full["ENet"],   w, label="ElasticNet Predicted",
       color=COLORS["elastic"],alpha=0.85, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(annual_full["YEAR"].astype(int), rotation=45, ha="right")
ax.set_xlabel("Year"); ax.set_ylabel("Annual Dengue Cases")
ax.set_title("Figure 8. Annual Dengue Cases — Actual vs. Predicted\n"
             "Bangladesh, 2008–2025", fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.axvline(len(annual_full)-1 - 0.5, color="red", ls="--", lw=1.5,
           label="2025 Forecast →")
ax.legend()
plt.tight_layout()
fig.savefig(f"{OUT}Fig8_Annual_Comparison.png")
plt.close()
print("Figure 8 saved  → Fig8_Annual_Comparison.png")

✔  Figure 8 saved  → Fig8_Annual_Comparison.png


## 14.  FIGURE 9 — Cross-Validation Fold Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 9. Time-Series CV RMSE per Fold (Training Set, Expanding Window)",
             fontsize=13, fontweight="bold")

folds = np.arange(1, 11)
for ax, (cv_scores, model_name, color) in zip(axes, [
    (cv_ridge, "Ridge Regression",     COLORS["ridge"]),
    (cv_enet,  "ElasticNet Regression", COLORS["elastic"]),
]):
    ax.bar(folds, cv_scores, color=color, alpha=0.8, edgecolor="white")
    ax.axhline(cv_scores.mean(), color="black", lw=1.8, ls="--",
               label=f"Mean = {cv_scores.mean():.3f}")
    ax.fill_between([0.5, 10.5],
                    cv_scores.mean()-cv_scores.std(),
                    cv_scores.mean()+cv_scores.std(),
                    alpha=0.15, color="black", label=f"±1 SD = {cv_scores.std():.3f}")
    ax.set_xticks(folds)
    ax.set_xlabel("Fold (chronological, expanding window)"); ax.set_ylabel("RMSE (log-scale)")
    ax.set_title(f"{model_name}")
    ax.legend()

plt.tight_layout()
fig.savefig(f"{OUT}Fig9_CrossValidation.png")
plt.close()
print(" Figure 9 saved  → Fig9_CrossValidation.png")

✔  Figure 9 saved  → Fig9_CrossValidation.png


## 15.  FIGURE 10 — Model Comparison Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(15, 10))
fig.suptitle("Figure 10. Model Comparison Dashboard — Ridge vs ElasticNet\n"
             "Bangladesh Dengue Forecast (2025)", fontsize=14, fontweight="bold")

gs = gridspec.GridSpec(2, 3, figure=fig, wspace=0.38, hspace=0.45)

metric_names = ["R²", "RMSE", "MAE", "MAPE(%)"]
ridge_test  = [r2_score(y_test_orig, ridge_test_pred),
               np.sqrt(mean_squared_error(y_test_orig, ridge_test_pred)),
               mean_absolute_error(y_test_orig, ridge_test_pred),
               mean_absolute_percentage_error(y_test_orig+1, ridge_test_pred+1)*100]
enet_test   = [r2_score(y_test_orig, enet_test_pred),
               np.sqrt(mean_squared_error(y_test_orig, enet_test_pred)),
               mean_absolute_error(y_test_orig, enet_test_pred),
               mean_absolute_percentage_error(y_test_orig+1, enet_test_pred+1)*100]

for i, (metric, r_val, e_val) in enumerate(zip(metric_names, ridge_test, enet_test)):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    bars = ax.bar(["Ridge", "ElasticNet"], [r_val, e_val],
                  color=[COLORS["ridge"], COLORS["elastic"]], alpha=0.85,
                  edgecolor="white", width=0.5)
    for bar, val in zip(bars, [r_val, e_val]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.02,
                f"{val:.3f}", ha="center", fontsize=11, fontweight="bold")
    ax.set_title(metric, fontsize=12)
    ax.set_ylabel(metric)

ax_radar = fig.add_subplot(gs[1, 1:], polar=True)
metrics_norm_ridge = [r2_score(y_test_orig, ridge_test_pred),
                      1 - np.sqrt(mean_squared_error(y_test_orig, ridge_test_pred)) / y_test_orig.max(),
                      1 - mean_absolute_error(y_test_orig, ridge_test_pred) / y_test_orig.max(),
                      1 - mean_absolute_percentage_error(y_test_orig+1, ridge_test_pred+1)]
metrics_norm_enet  = [r2_score(y_test_orig, enet_test_pred),
                      1 - np.sqrt(mean_squared_error(y_test_orig, enet_test_pred)) / y_test_orig.max(),
                      1 - mean_absolute_error(y_test_orig, enet_test_pred) / y_test_orig.max(),
                      1 - mean_absolute_percentage_error(y_test_orig+1, enet_test_pred+1)]
metrics_norm_ridge = [max(0, x) for x in metrics_norm_ridge]
metrics_norm_enet  = [max(0, x) for x in metrics_norm_enet]

labels_rad = ["R²", "1−NRMSE", "1−NMAE", "1−MAPE"]
N = len(labels_rad)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
for values, color, name in [
    (metrics_norm_ridge, COLORS["ridge"],   "Ridge"),
    (metrics_norm_enet,  COLORS["elastic"], "ElasticNet"),
]:
    vals = values + values[:1]
    ax_radar.plot(angles, vals, color=color, lw=2, label=name)
    ax_radar.fill(angles, vals, color=color, alpha=0.15)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(labels_rad, fontsize=11)
ax_radar.set_title("Normalized Performance\n(2025 Forecast)", fontsize=11, pad=15)
ax_radar.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

fig.savefig(f"{OUT}Fig10_Model_Comparison_Dashboard.png")
plt.close()
print(" Figure 10 saved → Fig10_Model_Comparison_Dashboard.png")

✔  Figure 10 saved → Fig10_Model_Comparison_Dashboard.png


## 16.  SAVE FULL PREDICTION TABLE

In [ ]:
pred_2025 = df_test[["YEAR","MONTH","MIN","MAX","HUMIDITY","RAINFALL","DENGUE"]].copy()
pred_2025.columns = ["Year","Month","Min_Temp","Max_Temp","Humidity","Rainfall","Actual_Cases"]
pred_2025["Month_Name"]         = pred_2025["Month"].apply(lambda m: MONTH_NAMES[m-1])
pred_2025["Ridge_Predicted"]    = ridge_test_pred.astype(int)
pred_2025["ElasticNet_Predicted"]= enet_test_pred.astype(int)
pred_2025["Ridge_Error"]        = pred_2025["Actual_Cases"] - pred_2025["Ridge_Predicted"]
pred_2025["ElasticNet_Error"]   = pred_2025["Actual_Cases"] - pred_2025["ElasticNet_Predicted"]
pred_2025["Ridge_AbsErr_Pct"]   = (pred_2025["Ridge_Error"].abs() / (pred_2025["Actual_Cases"]+1) * 100).round(1)
pred_2025["ENet_AbsErr_Pct"]    = (pred_2025["ElasticNet_Error"].abs() / (pred_2025["Actual_Cases"]+1) * 100).round(1)
pred_2025.to_csv(f"{OUT}2025_Prediction_Table.csv", index=False)

coef_df = pd.DataFrame({
    "Feature"           : feat_labels,
    "Ridge_Coefficient" : ridge.named_steps["ridge"].coef_,
    "ElasticNet_Coefficient": enet.named_steps["enet"].coef_,
})
coef_df.to_csv(f"{OUT}Model_Coefficients.csv", index=False)

## 17.  FINAL CONSOLE REPORT

In [ ]:
print("\n" + "="*70)
print("  FINAL MODEL SUMMARY")
print("="*70)
print(f"\n  Ridge Regression")
print(f"    Best alpha         : {best_alpha_ridge:.4f}")
print(f"    Train R²           : {r2_score(y_train_orig, ridge_train_pred):.4f}")
print(f"    Test  R² (2025)    : {r2_score(y_test_orig, ridge_test_pred):.4f}")
print(f"    Test  RMSE         : {np.sqrt(mean_squared_error(y_test_orig, ridge_test_pred)):,.1f}")
print(f"    Test  MAE          : {mean_absolute_error(y_test_orig, ridge_test_pred):,.1f}")

print(f"\n  ElasticNet Regression")
print(f"    Best alpha         : {best_alpha_enet:.4f}")
print(f"    Best l1_ratio      : {best_l1:.2f}")
print(f"    Non-zero coefs     : {np.sum(enet.named_steps['enet'].coef_ != 0)} / {len(FEATURES)}")
print(f"    Train R²           : {r2_score(y_train_orig, enet_train_pred):.4f}")
print(f"    Test  R² (2025)    : {r2_score(y_test_orig, enet_test_pred):.4f}")
print(f"    Test  RMSE         : {np.sqrt(mean_squared_error(y_test_orig, enet_test_pred)):,.1f}")
print(f"    Test  MAE          : {mean_absolute_error(y_test_orig, enet_test_pred):,.1f}")

print(f"\n{'─'*70}")
print(f"  2025 Monthly Prediction Summary")
print(f"{'─'*70}")
print(pred_2025[["Month_Name","Actual_Cases","Ridge_Predicted",
                  "Ridge_AbsErr_Pct","ElasticNet_Predicted","ENet_AbsErr_Pct"]
                ].to_string(index=False))


  FINAL MODEL SUMMARY

  Ridge Regression
    Best alpha         : 5.8570
    Train R²           : 0.7959
    Test  R² (2025)    : -1.1473
    Test  RMSE         : 9,000.7
    Test  MAE          : 5,222.8

  ElasticNet Regression
    Best alpha         : 0.0744
    Best l1_ratio      : 1.00
    Non-zero coefs     : 8 / 15
    Train R²           : 0.7491
    Test  R² (2025)    : -0.1001
    Test  RMSE         : 6,442.2
    Test  MAE          : 3,454.0

──────────────────────────────────────────────────────────────────────
  2025 Monthly Prediction Summary
──────────────────────────────────────────────────────────────────────
Month_Name  Actual_Cases  Ridge_Predicted  Ridge_AbsErr_Pct  ElasticNet_Predicted  ENet_AbsErr_Pct
       Jan          1161             2290              97.2                  2529            117.7
       Feb           374              625              66.9                   635             69.6
       Mar           336              370              10.1           